# exp-10 — check-major against panel-major, revisited on eleven checks

Compares the two answer shapes of exp-04 on the configuration the series
settled on — `A|B <- C_i <- D`, the eleven checks, Sonnet 5.5, `gold-v7` —
both run the same day by `fig-checklist-exp10`'s two entry points, identical
but for their name:

| shape | entry point | answer |
|---|---|---|
| check-major | `do-fig-checklist-cm` | `{<check>: [rows]}` |
| panel-major | `do-fig-checklist-pm` | `{"outputs": [{panel_label, panel_classes, <check>: {...}}]}` |

**The hypothesis**: check-major is non-inferior to panel-major on E1 and E2 —
the lower bound of the 95% interval of check-major minus panel-major above
−0.02. A panel-major answer is read per check — the shared panel row counts
once for every check, as a check-major panel counts once in each list.

**This notebook spends nothing.** The sessions are produced by

```bash
python experiments/exp-10-panel-major-revisited/run.py
```

Fixed in
[`thinking/experiments/exp-10-panel-major-revisited.md`](../../thinking/experiments/exp-10-panel-major-revisited.md).

## The gold

**`gold-v7`**. This cell pins the scorer to it and must run first.

In [ ]:
from soda_mmqc.gold import pin_gold

GOLD_TAG = "gold-v7"
pin_gold(GOLD_TAG)

In [ ]:
import importlib.util
import json
import os
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from soda_mmqc.config import gold_root
from soda_mmqc.core.run_layout import iter_leaves
from soda_mmqc.core.scoring import ANALYSIS_FILENAME, score_check, stale_examples
from soda_mmqc.reporting import (
    layer1_counts_by_example,
    layer2_counts,
    layer_s_counts_by_example,
    load_run_root,
    non_response_counts,
    plot_stacked_counts,
    scores_frame,
)
from soda_mmqc.reporting.styles import (
    LAYER1_ORDER,
    LAYER2_BINARY_ORDER,
    LAYER2_GRADED_ORDER,
    LAYER_S_ORDER,
)
from soda_mmqc.reporting.tables import LAYER1_OUTLIER_OUTCOMES, LAYER2_ERROR_OUTCOMES

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)


def _load(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


exp10 = _load("exp10_run", Path("../../experiments/exp-10-panel-major-revisited/run.py"))
panels = _load("exp04_panels", Path("../../experiments/exp-04-panel-major/panels.py"))
assert exp10.GOLD_TAG == GOLD_TAG, "the runner and this notebook name different gold"

CHECKS, CLASSIFY, MODEL = exp10.CHECKS, exp10.CLASSIFY, exp10.MODEL
IDENTIFY = "identify-panels"
CM, PM = "check-major", "panel-major"
#: Kept under the name the exp-07 code uses: the conditions, in order.
MODELS = [CM, PM]
SHORT = {m: m for m in MODELS}
#: The CLI's own background model, in every session's model usage.
BACKGROUND = "claude-haiku-5-5"
#: (a, b): a minus b. The hypothesis.
PAIRS = [(CM, PM)]
MARGIN = 0.02

#: One decision-like property per check: its decision, or what plays that part.
DECISIONS = {
    "micrograph-scale-bar": "scale_bar_defined_in_caption",
    "image-annotation-defined": "annotation_defined_in_caption",
    "single-channel-for-overlay": "all_channels_shown_separately",
    "individual-data-points": "decision",
    "error-bars-defined": "decision",
    "plot-axis-units": "decision",
    "plot-gap-labeling": "decision",
    "stat-significance-level": "decision",
    "stat-test": "decision",
    "replication-reporting": "decision",
    "panel-image-matches-caption": "image_matches_caption",
}
assert set(DECISIONS) == set(CHECKS)

#: Each condition: (run root, arm, checklist, entry point, shape).
#: Set EXP10_ROOT to `.../smoke` to exercise the notebook on the smoke test.
ROOT = Path(os.environ.get("EXP10_ROOT", "../../experiments/runs/exp-10-panel-major-revisited"))
SOURCES = {
    CM: (ROOT / "cm", "pinned", exp10.CHECKLIST, exp10.CM, "cm"),
    PM: (ROOT / "pm", "pinned", exp10.CHECKLIST, exp10.PM, "pm"),
}
GOLD = gold_root()
SEED = 20261023
RESAMPLES = 10_000


def model_leaves():
    """(condition, replicate, leaf) for every condition that has run."""
    for condition, (root, arm, *_) in SOURCES.items():
        if root.is_dir():
            for leaf_arm, replicate, leaf in iter_leaves(root):
                if leaf_arm == arm:
                    yield condition, replicate, leaf


def owning_check(value: str) -> str | None:
    for check in CHECKS:
        if value == check or value.startswith((f"{check}[", f"{check}.")):
            return check
    return None


def split_check(value: str) -> list[tuple[str, str]]:
    """(check, the value in the check's own terms), for both shapes.

    Check-major: `<check>[].x` -> `outputs[].x`; list `<check>` -> `outputs`.
    Panel-major: `outputs[].<check>.x` -> `outputs[].x`; list `<check>.<list>` ->
    `<list>`. The panel row itself -- `outputs[].panel_label`, the list
    `outputs` -- belongs to every check: one row per panel, read once per check.
    """
    m = re.match(r"^outputs\[\d*\]\.([^.\[]+)\.(.*)$", value)
    if m and m.group(1) in CHECKS:
        return [(m.group(1), "outputs[]." + m.group(2))]
    if value in ("outputs", "outputs[].panel_label") or re.match(r"^outputs\[\d*\]\.panel_label$", value):
        return [(c, value) for c in CHECKS]
    check = owning_check(value)
    if check is None:
        return []
    rest = value[len(check):]
    if rest.startswith("."):          # a nested list named by its path, panel-major
        return [(check, rest[1:])]
    return [(check, "outputs" + rest)]


def to_per_check(frame: pd.DataFrame, model: str) -> pd.DataFrame:
    """Rows in each check's own terms, plus `condition`; plot-axis-units' nested
    lists keep their names, as in exp-07 to exp-09."""
    frame = frame.drop(columns=[c for c in ("check", "condition") if c in frame])
    key = "property" if "property" in frame else "list_key"
    rows = []
    for record in frame.to_dict("records"):
        value = record[key]
        owners = split_check(value)
        if not owners and key == "list_key":
            owners = [("plot-axis-units", value)]      # check-major nested lists
        if not owners:
            if value.endswith("panel_classes"):
                continue                               # unscored
            raise ValueError(f"{key} owned by no check: {value}")
        for check, own in owners:
            rows.append({**record, "check": check, key: own, "condition": model})
    return pd.DataFrame(rows, columns=list(frame.columns) + ["check", "condition"])


def is_decision(check: str, prop: str) -> bool:
    return DECISIONS.get(check) == prop.removeprefix("outputs[].")


def bootstrap_mean(values, rng) -> tuple[float, float, float, int]:
    v = np.asarray([x for x in values if not pd.isna(x)], dtype=float)
    if not len(v):
        return (np.nan, np.nan, np.nan, 0)
    boots = rng.choice(v, size=(RESAMPLES, len(v)), replace=True).mean(axis=1)
    return (v.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5), len(v))


for condition, (root, arm, checklist, entry, shape) in SOURCES.items():
    print(f"{condition:<12} {checklist} {entry:<20} {root.resolve() / arm}")

## Score every leaf that is not scored yet

In [ ]:
# Compared by content (core.scoring.stale_examples), not by file time:
# a branch checkout rewrites predictions, and extracting a pinned gold
# snapshot rewrites gold, with new mtimes and the same bytes.
stale = {}
for model, replicate, leaf in model_leaves():
    if not (leaf / ANALYSIS_FILENAME).is_file():
        score_check(SOURCES[model][2], SOURCES[model][3], leaf, save=True)
        print(f"scored {model} rep-{replicate:02d}")
        continue
    changed = stale_examples(SOURCES[model][2], SOURCES[model][3], leaf)
    if changed:
        stale[leaf] = changed

if stale:
    raise RuntimeError(
        "predictions or gold differ from what their analysis scored in "
        f"{len(stale)} leaf/leaves; delete those analysis.json and re-run "
        "this cell:\n  "
        + "\n  ".join(f"{leaf} ({len(examples)} example(s))" for leaf, examples in stale.items())
    )
print("all leaves scored, none stale")

## 0. Dispatch, read first

Per session, from `skill_trace.json`: how many of the eleven checks were
invoked and which were not; how often `classify-panels` was invoked, and
**whether it came before the first check** — the entry point classifying the
panels itself, which exp-04 found under panel-major; whether the checks ran in
D's order; any skill outside the closure.

In [ ]:
def read_json(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None


rows = []
for model, replicate, leaf in model_leaves():
    for trace in leaf.rglob("intermediates/skill_trace.json"):
        skills = [e.get("skill") for e in read_json(trace) or [] if isinstance(e, dict) and e.get("skill")]
        called = list(dict.fromkeys(s for s in skills if s in CHECKS))
        rows.append({
            "condition": model, "replicate": replicate,
            "example": str(trace.parent.parent.relative_to(leaf)),
            "n_checks": len(called),
            **{c: c in called for c in CHECKS},
            "classify_calls": skills.count(CLASSIFY),
            "classify_before_checks": CLASSIFY in skills and (not called or skills.index(CLASSIFY)
                                                              < skills.index(called[0])),
            "identify_calls": skills.count(IDENTIFY),
            "in_order": called == [c for c in CHECKS if c in called],
            "spontaneous": tuple(sorted(set(skills) - {SOURCES[model][3], CLASSIFY, IDENTIFY, *CHECKS})),
        })
sessions = pd.DataFrame(rows)
sessions["dispatch"] = np.select([sessions["n_checks"] == len(CHECKS), sessions["n_checks"] == 0],
                                 ["all eleven", "no check"], "partial")
print(sessions.groupby(["condition", "dispatch"]).size().unstack(fill_value=0).reindex(MODELS))
print()
print("checks invoked in partial sessions:")
print(sessions[sessions["dispatch"] == "partial"].groupby(["condition", "n_checks"]).size())

In [ ]:
# Per check, in D's order: the share of sessions that invoked it.
sessions.groupby("condition")[list(CHECKS)].mean().T[[m for m in MODELS if m in set(sessions["condition"])]]

In [ ]:
for col in ("classify_calls", "classify_before_checks"):
    print(col)
    print(sessions.groupby(["condition", col]).size().unstack(fill_value=0))
print("in D's order:", sessions.groupby("condition")["in_order"].agg(["sum", "count"]).to_dict("index"))
spont = sessions.explode("spontaneous").dropna(subset=["spontaneous"])
print("spontaneous skills:", "none" if spont.empty else spont.groupby(["condition", "spontaneous"]).size().to_dict())

## The model that answered: fallback and refusals

From each session's `tool_audit.json`: the models in its usage and, per
assistant message, the model that wrote it. Claude Code re-runs a request its
safety classifier flags on a fallback model and continues there (Sonnet
5.5: cybersecurity to Sonnet 5; biology ends in a refusal). Allowed and
recorded, as in exp-07. The CLI's background model, `claude-haiku-5-5`, is
set apart.

In [ ]:
rows = []
for model, replicate, leaf in model_leaves():
    for audit_path in leaf.rglob("intermediates/tool_audit.json"):
        audit = read_json(audit_path) or {}
        usage = audit.get("usage") or {}
        per_model = usage.get("model_usage") or {}
        others = sorted(m for m in per_model if m not in (MODEL, BACKGROUND))
        sequence = []
        for turn in audit.get("turns") or []:
            if not sequence or sequence[-1] != turn["model"]:
                sequence.append(turn["model"])
        tokens = usage.get("usage") or {}
        rows.append({
            "condition": model, "replicate": replicate,
            "example": str(audit_path.parent.parent.relative_to(leaf)),
            "other_models": tuple(others),
            "other_output_tokens": sum((per_model[m] or {}).get("outputTokens") or 0 for m in others),
            "requested_output_tokens": (per_model.get(MODEL) or {}).get("outputTokens") or 0,
            "sequence": " -> ".join(sequence),
            "stop_reason": usage.get("stop_reason"),
            "terminal_reason": usage.get("terminal_reason"),
            "cost": usage.get("total_cost_usd"),
            "background_cost": (per_model.get(BACKGROUND) or {}).get("costUSD"),
            "turns": usage.get("num_turns"),
            "seconds": (usage.get("duration_ms") or 0) / 1000,
            "input_tokens": sum(tokens.get(k) or 0 for k in
                                ("input_tokens", "cache_creation_input_tokens", "cache_read_input_tokens")),
            "output_tokens": tokens.get("output_tokens") or 0,
            "concurrency": (audit.get("session") or {}).get("concurrency"),
        })
usage = pd.DataFrame(rows)
usage["fallback"] = usage["other_models"].map(bool)
print("sessions with another model:")
print(usage.groupby("condition")["fallback"].agg(["sum", "count"]).reindex(MODELS))
print()
print("to which:", usage[usage["fallback"]].groupby(["condition", "other_models"]).size().to_dict() or "none")
print("stop reasons:", usage.groupby(["condition", "stop_reason"], dropna=False).size().to_dict())
usage[usage["fallback"]][["condition", "replicate", "example", "other_models", "requested_output_tokens",
                          "other_output_tokens", "sequence"]]

### The examples that triggered a fallback

Per figure: which model switched, to which, in how many of its replicates.
A biology flag follows the figure's subject — `s44318-026-00715-1`, for
instance, is about mycobacteria (*M. tuberculosis*, *M. smegmatis*), which are
pathogens — so the same figure can switch again in another replicate, and not
every time.

In [ ]:
fallback_by_example = (
    usage.assign(switched=usage["fallback"])
    .groupby(["example", "condition"])
    .agg(replicates_switched=("switched", "sum"), replicates_run=("switched", "size"),
         fallback_models=("other_models", lambda s: sorted({m for t in s for m in t})),
         output_tokens_on_fallback=("other_output_tokens", "sum"))
    .reset_index()
)
fallback_by_example = fallback_by_example[fallback_by_example["replicates_switched"] > 0]
fallback_by_example if len(fallback_by_example) else "no example triggered a fallback"

## One row per property, per replicate, per example

In [ ]:
frames = {"scores": [], "layer_s": [], "layer_1": [], "layer_2": [], "non_response": []}
for model, (root, arm, checklist, entry, _) in SOURCES.items():
    if not root.is_dir():
        continue
    runs = load_run_root(root, checklist=checklist, check=entry)
    only = lambda frame: frame[frame["arm"] == arm]
    frames["scores"].append(to_per_check(only(scores_frame(runs)), model))
    frames["layer_s"].append(to_per_check(only(layer_s_counts_by_example(runs)), model))
    frames["layer_1"].append(to_per_check(only(layer1_counts_by_example(runs)), model))
    frames["layer_2"].append(to_per_check(only(layer2_counts(runs)), model))
    frames["non_response"].append(to_per_check(only(non_response_counts(runs)), model))
scores, layer_s, layer_1, layer_2, non_response = (pd.concat(frames[k], ignore_index=True) for k in frames)
PRESENT = [m for m in MODELS if m in set(scores["condition"])]
PLOTTED = PRESENT
print(f"{len(scores)} score rows; replicates per model:",
      scores.groupby("condition")["replicate"].nunique().to_dict())

## Failures and empty answers

A failed session loses all eleven checks.

In [ ]:
print(non_response[non_response["list_key"] == "outputs"]
      .groupby(["check", "condition"])["empty"].sum().unstack(fill_value=0))
wanted = json.loads((Path("../../soda_mmqc/data/checklist") / SOURCES[CM][2] / SOURCES[CM][3] / "benchmark.json")
                    .read_text(encoding="utf-8"))["examples"]
missing = [{"condition": model, "replicate": rep,
            "without_prediction": sum(not (leaf / ex / "prediction.json").is_file() for ex in wanted)}
           for model, rep, leaf in model_leaves()]
pd.DataFrame(missing).groupby("condition")["without_prediction"].sum()

## The outcomes, model beside model

In [ ]:
plot_stacked_counts(
    (layer_s[layer_s['list_key'] == 'outputs'])[lambda f: f['condition'].isin(PLOTTED)], group="check", category="outcome", series="condition",
    order=LAYER_S_ORDER, series_order=PLOTTED, ylabel="rows",
    title="Layer S row outcomes, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    (layer_1)[lambda f: f['condition'].isin(PLOTTED)], group="check", category="layer1", series="condition",
    order=LAYER1_ORDER, series_order=PLOTTED, ylabel="instances",
    title="Layer 1 applicability calls, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    (layer_2[layer_2['layer2'].isin(LAYER2_BINARY_ORDER)])[lambda f: f['condition'].isin(PLOTTED)], group="check", category="layer2", series="condition",
    order=LAYER2_BINARY_ORDER, series_order=PLOTTED, ylabel="instances",
    title="Layer 2, binary properties, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    (layer_2[layer_2['layer2'].isin(LAYER2_GRADED_ORDER)])[lambda f: f['condition'].isin(PLOTTED)], group="check", category="layer2", series="condition",
    order=LAYER2_GRADED_ORDER, series_order=PLOTTED, ylabel="instances",
    title="Layer 2, graded and multiclass properties, summed over examples and replicates",
)

## Layer S per check, and panels across the eleven lists

Per check, `correct / (correct + missing)` and
`correct / (correct + missing + spurious)`: per-figure rates, replicates
pooled, mean over figures with a bootstrap interval. Then, per figure, the
panels stated against the gold's: spurious and missing in both shapes; partly
missing — named in some check lists but not others — exists only check-major.

In [ ]:
def per_example_rate(frame, category, numerator, denominator=None, by=("check",)):
    counts = (frame.groupby(["condition", *by, "example", category])["count"].sum()
              .unstack(fill_value=0))
    num = counts[[c for c in numerator if c in counts]].sum(axis=1)
    den = counts[[c for c in (denominator or counts.columns) if c in counts]].sum(axis=1)
    return (num / den.where(den > 0)).rename("rate").reset_index()


rng = np.random.default_rng(SEED)
rows = []
rows_s = layer_s[layer_s["list_key"] == "outputs"]
for label, den in (("found", ("correct_row", "missing_row")),
                   ("both errors", ("correct_row", "missing_row", "spurious_row"))):
    r = per_example_rate(rows_s, "outcome", ("correct_row",), den)
    for (cond, check), sub in r.groupby(["condition", "check"]):
        mean, low, high, n = bootstrap_mean(sub["rate"], rng)
        rows.append({"rate": label, "condition": cond, "check": check, "mean": mean,
                     "ci_low": low, "ci_high": high, "n_examples": n})
layer_s_rates = pd.DataFrame(rows)
layer_s_rates.pivot_table(index="check", columns=["rate", "condition"], values="mean").reindex(CHECKS)

In [ ]:
rows = []
for model, replicate, leaf in model_leaves():
    for prediction in leaf.rglob("prediction.json"):
        example = str(prediction.parent.relative_to(leaf))
        gold = json.loads((GOLD / example / "checks" / exp10.CM / "expected_output.json")
                          .read_text(encoding="utf-8"))
        labels = [r["panel_label"] for r in gold[CHECKS[0]]]
        rows.append({"condition": model, "replicate": replicate, "example": example,
                     **panels.panel_counts(labels, json.loads(prediction.read_text()),
                                           SOURCES[model][4], CHECKS)})
panel_rows = pd.DataFrame(rows)
totals = panel_rows.groupby("condition")[["spurious_panels", "missing_panels",
                                         "partly_missing_panels", "gold_panels"]].sum()
totals["sessions"] = panel_rows.groupby("condition").size()
rng = np.random.default_rng(SEED)
per_figure_spurious = panel_rows.groupby(["condition", "example"])["spurious_panels"].mean()
for cond in PRESENT:
    mean, low, high, n = bootstrap_mean(per_figure_spurious[cond], rng)
    totals.loc[cond, "spurious per figure"] = f"{mean:.3f} [{low:.3f}, {high:.3f}]"
totals.reindex(PRESENT).fillna(0)

## Layer 1, per check

In [ ]:
rng = np.random.default_rng(SEED)
r = per_example_rate(layer_1, "layer1", ("correct_applicable", "correct_NA"))
rows = []
for (cond, check), sub in r.groupby(["condition", "check"]):
    mean, low, high, n = bootstrap_mean(sub["rate"], rng)
    rows.append({"condition": cond, "check": check, "mean": mean, "ci_low": low,
                 "ci_high": high, "n_examples": n})
layer_1_rates = pd.DataFrame(rows)
layer_1_rates.pivot_table(index="check", columns="condition", values="mean").reindex(CHECKS)[PRESENT]

## Layer 2, per check and per property

`mean_score` on applicable instances: per figure, averaged over replicates;
then the mean over figures with a bootstrap interval.

In [ ]:
rng = np.random.default_rng(SEED)
per_fig = (scores.dropna(subset=["mean_score"])
           .groupby(["condition", "check", "property", "example"])["mean_score"].mean().reset_index())
rows = []
for (cond, check, prop), sub in per_fig.groupby(["condition", "check", "property"]):
    mean, low, high, n = bootstrap_mean(sub["mean_score"], rng)
    rows.append({"condition": cond, "check": check, "property": prop, "mean": mean,
                 "ci_low": low, "ci_high": high, "n_examples": n})
layer_2_props = pd.DataFrame(rows)
layer_2_props["check"] = pd.Categorical(layer_2_props["check"], CHECKS, ordered=True)
layer_2_props.pivot_table(index=["check", "property"], columns="condition",
                          values="mean", observed=True)[PRESENT]

## Layer 2, each check's decision

One decision-like property per check, read on its own rather than averaged
with the check's other properties.

In [ ]:
decisions = layer_2_props[[is_decision(c, p) for c, p in
                           zip(layer_2_props["check"], layer_2_props["property"])]].copy()
decisions["check"] = decisions["check"].astype(str)
decisions.pivot_table(index="check", columns="condition",
                      values=["mean", "ci_low", "ci_high", "n_examples"]).reindex(CHECKS).round(3)

In [ ]:
import plotly.graph_objects as go

# One group per check, one bar per arrangement from 0 to its mean, the whisker
# its 95% bootstrap interval. The axis starts at 0, so differences of a
# hundredth look as small as they are. Each arrangement keeps its colour, and
# a pattern, so identity does not rest on colour alone.
STYLE = {CM: ("#2a78d6", ""), PM: ("#eb6834", "/")}

fig = go.Figure()
for cond in PLOTTED:
    sub = decisions[decisions["condition"] == cond].set_index("check").reindex(CHECKS).reset_index()
    colour, pattern = STYLE.get(cond, ("#6b6b6b", ""))
    fig.add_trace(go.Bar(
        x=sub["mean"],
        y=[f"{c} · {DECISIONS[c]}" for c in sub["check"]],
        orientation="h",
        name=SHORT[cond],
        marker=dict(color=colour, line=dict(color="white", width=1),
                    pattern=dict(shape=pattern, fgcolor="white", size=6, solidity=0.15)),
        error_x=dict(type="data", symmetric=False, array=sub["ci_high"] - sub["mean"],
                     arrayminus=sub["mean"] - sub["ci_low"], color="#3d3d3a", thickness=1.5, width=3),
        customdata=sub[["check", "ci_low", "ci_high", "n_examples"]].to_numpy(),
        hovertemplate=("<b>%{customdata[0]}</b><br>" + SHORT[cond]
                       + "<br>mean %{x:.3f} [%{customdata[1]:.3f}, %{customdata[2]:.3f}]"
                       + "<br>%{customdata[3]} figures<extra></extra>"),
    ))
fig.update_layout(
    title="Layer 2, each check's decision",
    template="plotly_white",
    barmode="group",
    bargap=0.25,
    bargroupgap=0.08,
    height=70 * len(CHECKS) + 160,
    margin=dict(l=20, r=20, t=60, b=80),
    legend=dict(orientation="h", yanchor="top", y=-0.08, xanchor="center", x=0.5, traceorder="normal"),
    hoverlabel=dict(align="left"),
)
fig.update_xaxes(title="mean_score, mean over figures (95% bootstrap interval)",
                 range=[0, 1.02], dtick=0.1, showgrid=True, gridcolor="#ebebea", zeroline=False)
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.show()

## The comparison: E1, E2 and E3

Per figure, replicates averaged (counts pooled for E1 and E2), check-major
minus panel-major: **better**, **equivalent** (within ±0.02), **both** or
**undecided**, by exp-07's rule.

In [ ]:
def per_figure_endpoints() -> pd.DataFrame:
    e1 = per_example_rate(rows_s.assign(all="all"), "outcome", ("correct_row",),
                          ("correct_row", "missing_row", "spurious_row"), by=("all",))
    e1 = e1.assign(endpoint="E1 layer S")
    e2 = per_example_rate(layer_1.assign(all="all"), "layer1", ("correct_applicable", "correct_NA"),
                          by=("all",)).assign(endpoint="E2 layer 1")
    d = scores.dropna(subset=["mean_score"])
    d = d[[is_decision(c, p) for c, p in zip(d["check"], d["property"])]]
    e3 = (d.groupby(["condition", "check", "example"])["mean_score"].mean().rename("rate").reset_index()
          .assign(endpoint=lambda f: "E3 " + f["check"]))
    keep = ["condition", "endpoint", "example", "rate"]
    return pd.concat([e1[keep], e2[keep], e3[keep]], ignore_index=True)


def classify(low: float, high: float) -> str:
    differs = low > 0 or high < 0
    within = low >= -MARGIN and high <= MARGIN
    return {(True, True): "both", (True, False): "better", (False, True): "equivalent",
            (False, False): "undecided"}[(differs, within)]


endpoints = per_figure_endpoints()
ENDPOINTS = ["E1 layer S", "E2 layer 1"] + [f"E3 {c}" for c in CHECKS]
rng = np.random.default_rng(SEED)
rows = []
for a, b in PAIRS:
    if a not in PRESENT or b not in PRESENT:
        continue
    for endpoint in ENDPOINTS:
        sub = endpoints[endpoints["endpoint"] == endpoint].pivot_table(
            index="example", columns="condition", values="rate")
        if a not in sub or b not in sub:
            continue
        diff = (sub[a] - sub[b]).dropna().to_numpy()
        if not len(diff):
            continue
        boots = rng.choice(diff, size=(RESAMPLES, len(diff)), replace=True).mean(axis=1)
        low, high = np.percentile(boots, [2.5, 97.5])
        verdict = classify(low, high)
        if verdict in ("better", "both"):
            verdict += f": {SHORT[a] if diff.mean() > 0 else SHORT[b]}"
        rows.append({"pair": f"{SHORT[a]} - {SHORT[b]}", "endpoint": endpoint,
                     "difference": diff.mean(), "ci_low": low, "ci_high": high,
                     "n_figures": len(diff), "verdict": verdict})
comparison = pd.DataFrame(rows)
comparison.pivot_table(index="endpoint", columns="pair", values="verdict", aggfunc="first").reindex(ENDPOINTS)

In [ ]:
comparison.round(4)

## The hypothesis: check-major non-inferior to panel-major

The paired difference over figures, check-major minus panel-major, on E1 and
E2: **non-inferior** if the lower bound of its 95% interval is above −0.02;
**inferior** if the interval is wholly below 0 and reaches past −0.02;
**inconclusive** otherwise. The eleven decisions are classified the same way,
as secondary endpoints, not part of the verdict.

In [ ]:
def non_inferiority(low: float, high: float) -> str:
    if low > -MARGIN:
        return "non-inferior"
    if high < 0:
        return "inferior"
    return "inconclusive"


rng = np.random.default_rng(SEED)
rows = []
for endpoint in ENDPOINTS:
    sub = endpoints[endpoints["endpoint"] == endpoint].pivot_table(
        index="example", columns="condition", values="rate")
    if CM not in sub or PM not in sub:
        continue
    diff = (sub[CM] - sub[PM]).dropna().to_numpy()
    boots = rng.choice(diff, size=(RESAMPLES, len(diff)), replace=True).mean(axis=1)
    low, high = np.percentile(boots, [2.5, 97.5])
    rows.append({"endpoint": endpoint, "check-major": sub[CM].mean(),
                 "panel-major": sub[PM].mean(), "difference": diff.mean(), "ci_low": low,
                 "ci_high": high, "n_figures": len(diff), "verdict": non_inferiority(low, high),
                 "primary": not endpoint.startswith("E3")})
hypothesis = pd.DataFrame(rows)
primary = hypothesis[hypothesis["primary"]]
if len(primary) == 2:
    verdicts = set(primary["verdict"])
    overall = ("NON-INFERIOR" if verdicts == {"non-inferior"} else
               "INFERIOR" if "inferior" in verdicts else "INCONCLUSIVE")
    print(f"{CM} against {PM}: {overall}")
hypothesis.round(4)

### The two shapes side by side

E1 and E2 per shape, and on the sessions that invoked all eleven checks only
— which separates the shape's effect on dispatch from its effect on the checks.

In [ ]:
full = set(map(tuple, sessions[sessions["dispatch"] == "all eleven"][
    ["condition", "replicate", "example"]].itertuples(index=False)))
keep = lambda f: f[[k in full for k in zip(f["condition"], f["replicate"], f["example"])]]
side = pd.DataFrame({
    "E1 layer S": per_example_rate(rows_s.assign(all="all"), "outcome", ("correct_row",),
                                   ("correct_row", "missing_row", "spurious_row"), by=("all",))
                  .groupby("condition")["rate"].mean(),
    "E2 layer 1": per_example_rate(layer_1.assign(all="all"), "layer1",
                                   ("correct_applicable", "correct_NA"), by=("all",))
                  .groupby("condition")["rate"].mean(),
    "E1, complete sessions": per_example_rate(keep(rows_s).assign(all="all"), "outcome", ("correct_row",),
                                              ("correct_row", "missing_row", "spurious_row"), by=("all",))
                             .groupby("condition")["rate"].mean(),
    "E2, complete sessions": per_example_rate(keep(layer_1).assign(all="all"), "layer1",
                                              ("correct_applicable", "correct_NA"), by=("all",))
                             .groupby("condition")["rate"].mean(),
    "all eleven checks": sessions.groupby("condition")["dispatch"].apply(lambda s: (s == "all eleven").mean()),
}).reindex(PRESENT)
side.round(4)

## Cost and time per figure

The CLI's cost estimates, at concurrency 50.
The background model's share is included in `cost` and shown apart. Token
counts are given so that cost can be recomputed at one price list.

In [ ]:
rng = np.random.default_rng(SEED)
per_figure = usage.groupby(["condition", "example"]).mean(numeric_only=True).reset_index()
rows = []
for cond, sub in per_figure.groupby("condition"):
    for q in ("cost", "background_cost", "input_tokens", "output_tokens", "turns", "seconds"):
        mean, low, high, n = bootstrap_mean(sub[q], rng)
        rows.append({"condition": cond, "quantity": q, "mean": mean, "ci_low": low,
                     "ci_high": high, "n_figures": n})
print("concurrency recorded:", usage["concurrency"].value_counts().to_dict())
print("total cost:", usage.groupby("condition")["cost"].sum().round(2).to_dict(),
      f"= ${usage['cost'].sum():.2f} over {len(usage)} sessions")
pd.DataFrame(rows).pivot_table(index="quantity", columns="condition", values="mean")[PRESENT]

In [ ]:
# Cost per figure: per figure averaged over replicates, mean over figures with
# a 95% bootstrap interval. The CLI's estimates, background model included.
rng = np.random.default_rng(SEED)
cost_rows = []
for cond in PLOTTED:
    sub = usage[usage["condition"] == cond].groupby("example")["cost"].mean()
    mean, low, high, n = bootstrap_mean(sub, rng)
    cost_rows.append({"condition": cond, "mean": mean, "ci_low": low, "ci_high": high, "n_figures": n})
cost = pd.DataFrame(cost_rows)

fig = go.Figure()
for row in cost.itertuples():
    colour, pattern = STYLE.get(row.condition, ("#6b6b6b", ""))
    fig.add_trace(go.Bar(
        x=[row.mean], y=[SHORT[row.condition]], orientation="h", name=SHORT[row.condition],
        marker=dict(color=colour, line=dict(color="white", width=1),
                    pattern=dict(shape=pattern, fgcolor="white", size=6, solidity=0.15)),
        error_x=dict(type="data", symmetric=False, array=[row.ci_high - row.mean],
                     arrayminus=[row.mean - row.ci_low], color="#3d3d3a", thickness=1.5, width=3),
        hovertemplate=(f"<b>{SHORT[row.condition]}</b><br>${row.mean:.3f} per figure "
                       f"[{row.ci_low:.3f}, {row.ci_high:.3f}]<br>{row.n_figures} figures<extra></extra>"),
    ))
# Each value just past its whisker, so the label never sits on it.
for row in cost.itertuples():
    fig.add_annotation(x=row.ci_high, y=SHORT[row.condition], text=f"${row.mean:.3f}",
                       showarrow=False, xanchor="left", xshift=8, font=dict(color="#3d3d3a"))
fig.update_layout(
    title="Cost per figure (USD, the CLI's estimate)",
    template="plotly_white", showlegend=False, bargap=0.35,
    height=90 * len(PLOTTED) + 140, margin=dict(l=20, r=60, t=60, b=60),
)
fig.update_xaxes(title="USD per figure, mean over figures (95% bootstrap interval)",
                 range=[0, cost["ci_high"].max() * 1.2], showgrid=True, gridcolor="#ebebea",
                 zeroline=False, tickprefix="$")
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.show()

## Sessions that did not invoke every check

Each beside the same figure's sessions of the same model that invoked all
eleven: layer S and layer 1 pooled over the checks, errors on the eleven
decisions counted (not averaged across properties), and size.

In [ ]:
KEY = ["condition", "replicate", "example"]


def session_rates() -> pd.DataFrame:
    s = (rows_s.groupby(KEY + ["outcome"])["count"].sum().unstack(fill_value=0))
    out = pd.DataFrame({
        "S_both": s["correct_row"] / s[["correct_row", "missing_row", "spurious_row"]].sum(axis=1),
        "missing_rows": s["missing_row"], "spurious_rows": s["spurious_row"],
    })
    l1 = layer_1.groupby(KEY + ["layer1"])["count"].sum().unstack(fill_value=0)
    out["layer_1"] = l1[["correct_applicable", "correct_NA"]].sum(axis=1) / l1.sum(axis=1)
    d = scores.dropna(subset=["mean_score"])
    d = d[[is_decision(c, p) for c, p in zip(d["check"], d["property"])]]
    d = d.assign(errors=(1 - d["mean_score"]) * d["n_scored"])
    out["decision_errors"] = d.groupby(KEY)["errors"].sum()
    out["decision_instances"] = d.groupby(KEY)["n_scored"].sum()
    out = out.join(usage.set_index(KEY)[["output_tokens", "turns", "cost"]])
    return out.join(sessions.set_index(KEY)[["n_checks", "dispatch"]]).reset_index()


per_session = session_rates()
columns = ["n_checks", "S_both", "missing_rows", "spurious_rows", "layer_1", "decision_errors",
           "decision_instances", "output_tokens", "turns", "cost"]
rows = []
for _, z in per_session[per_session["dispatch"] != "all eleven"].iterrows():
    same = per_session[(per_session["condition"] == z["condition"]) & (per_session["example"] == z["example"])
                       & (per_session["dispatch"] == "all eleven")]
    rows.append({"condition": z["condition"], "example": z["example"], "session": z["dispatch"],
                 "replicate": z["replicate"], **z[columns].to_dict()})
    rows.append({"condition": z["condition"], "example": z["example"],
                 "session": f"all eleven (mean of {len(same)})", "replicate": None,
                 **(same[columns].mean().to_dict() if len(same) else {})})
short_dispatch = pd.DataFrame(rows)
short_dispatch.set_index(["condition", "example", "session"]).round(3) if rows else "every session invoked all eleven"

## Error analysis: the candidates, on the decisions

Every layer-1 applicability error and layer-2 disagreement on the eleven
decisions, per model, and those recurring in at least two of the three
replicates of a model, for the manual sorting — **model error**, **gold
error** or **ambiguous**. Written to
`experiments/exp-10-panel-major-revisited/error_analysis_candidates.csv` (not committed) when
run on the full run; annotate a copy.

In [ ]:
def row_index(path: str) -> int | None:
    head = path.split("].", 1)[0]
    return int(head.rsplit("[", 1)[1]) if "[" in head else None


rows = []
for model, replicate, leaf in model_leaves():
    analysis = json.loads((leaf / ANALYSIS_FILENAME).read_text(encoding="utf-8"))
    for record in analysis.get("flat", []):
        example = (record.get("metadata") or {}).get("source")
        gold = record.get("expected_output") or {}
        for inst in (record.get("analysis") or {}).get("instances", []):
            owners = split_check(inst["leaf_property"])
            if len(owners) != 1:
                continue
            check, prop = owners[0]
            if not is_decision(check, prop):
                continue
            l1, l2, score = inst.get("layer1"), inst.get("layer2"), inst.get("score")
            list_error = l1 is None and l2 is None and score is not None and score < 1
            if l1 not in LAYER1_OUTLIER_OUTCOMES and l2 not in LAYER2_ERROR_OUTCOMES and not list_error:
                continue
            i = row_index(inst["path"])
            gold_rows = gold.get(check) or gold.get("outputs") or []
            panel = gold_rows[i].get("panel_label") if i is not None and i < len(gold_rows) else None
            rows.append({
                "condition": model, "replicate": replicate, "example": example, "check": check,
                "panel": panel, "property": prop,
                "outcome": l1 if l1 in LAYER1_OUTLIER_OUTCOMES else l2 if l2 in LAYER2_ERROR_OUTCOMES
                else f"list score {score:.2f}",
                "exp_value": json.dumps(inst.get("exp_value"), ensure_ascii=False),
                "pred_value": json.dumps(inst.get("pred_value"), ensure_ascii=False),
            })
errors = pd.DataFrame(rows)
print(f"{len(errors)} decision error instances")
(errors.groupby(["check", "condition"]).size().unstack(fill_value=0).reindex(CHECKS)
 .reindex(columns=PRESENT).fillna(0).astype(int)) if rows else errors

In [ ]:
recurring = (errors.groupby(["condition", "example", "check", "panel", "property", "exp_value"])
             .agg(sessions_wrong=("replicate", "nunique"),
                  most_common_pred=("pred_value", lambda s: Counter(s).most_common(1)[0][0]),
                  outcome=("outcome", lambda s: Counter(s).most_common(1)[0][0]))
             .reset_index()
             .sort_values(["sessions_wrong", "check"], ascending=[False, True])) if len(errors) else errors
# Wrong in the same way across models is a gold candidate before it is a model one.
if len(recurring):
    models_wrong = recurring.groupby(["example", "check", "panel", "property"])["condition"].nunique()
    recurring["models_wrong"] = [models_wrong[k] for k in
                                 zip(recurring["example"], recurring["check"], recurring["panel"], recurring["property"])]
    recurring["sorting"], recurring["note"] = "", ""
    out = Path("../../experiments/exp-10-panel-major-revisited/error_analysis_candidates.csv")
    picked = recurring[recurring["sessions_wrong"] >= 2]
    if "EXP10_ROOT" in os.environ:
        print(f"{len(picked)} recurring; not written -- EXP10_ROOT points away from the full run")
    else:
        picked.to_csv(out, index=False)
        print(f"{len(picked)} decision disagreements wrong in at least 2 replicates, written to {out.name}")
recurring.head(40)

## What this does not do

It does not sort the disagreements: that is the manual error analysis,
recorded in
[`thinking/experiments/exp-10-panel-major-revisited.md`](../../thinking/experiments/exp-10-panel-major-revisited.md).